In [3]:
import spacy
import requests
import re
import json
from difflib import SequenceMatcher

# Load spaCy's English model
nlp = spacy.load("en_core_web_lg")

def string_similarity(a, b):
    """Compute string similarity score using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_wikidata(entity):
    """Search Wikidata for an entity and return the best match based on relevance."""
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            best_match = None
            highest_score = 0.0
            
            for result in data["search"]:
                label = result.get("label", "")
                description = result.get("description", "")
                wikidata_id = result["id"]

                # Compute similarity score
                similarity = string_similarity(entity, label)

                # Final score
                final_score = similarity
                
                if final_score > highest_score:
                    highest_score = final_score
                    best_match = {
                        "entity": entity,
                        "wikidata_id": wikidata_id,
                        "label": label,
                        "description": description,
                        "wikidata_url": f"https://www.wikidata.org/wiki/{wikidata_id}",
                        "relevance_score": round(final_score, 3)
                    }
            
            return best_match  # Return only the best match
    return None

def search_wikidata_property(relation):
    """Search Wikidata for a property related to the relation (verb)."""
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "type": "property",
        "search": relation
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            best_match = data["search"][0]
            return {
                "relation": relation,
                "wikidata_property_id": best_match["id"],
                "label": best_match.get("label", ""),
                "description": best_match.get("description", ""),
                "wikidata_property_url": f"https://www.wikidata.org/wiki/Property:{best_match['id']}"
            }
    return None

def extract_entities(text):
    """Extract entities using both NER and keyword extraction."""
    doc = nlp(text)
    entities = set(ent.text for ent in doc.ents)  # Extract named entities
    
    # Extract additional keywords (noun chunks)
    for chunk in doc.noun_chunks:
        clean_chunk = chunk.text.strip()
        if clean_chunk and len(clean_chunk) > 2:  # Avoid short words
            entities.add(chunk.text)

    return list(entities)

def extract_relationships(text):
    """Extract relationships between entities using dependency parsing."""
    doc = nlp(text)
    relationships = []
    
    for token in doc:
        if token.pos_ == "VERB":  # Identify verbs that indicate relationships
            subject = None
            objects = []
            
            # Find subject (nsubj)
            for child in token.children:
                if child.dep_ in {"nsubj", "nsubjpass"}:
                    subject = child.text
            
            # Find direct objects (dobj) or prepositional objects (pobj)
            for child in token.children:
                if child.dep_ in {"dobj", "pobj"}:
                    objects.append(child.text)
            
            # Handle conjunction (e.g., "developed A and influenced B")
            for child in token.conjuncts:
                if child.pos_ == "VERB":
                    objects.append(child.text)
            
            if subject and objects:
                for obj in objects:
                    relation = {
                        "subject": subject,
                        "relation": token.text,
                        "object": obj
                    }
                    # Try to find the Wikidata property for the relation
                    wikidata_prop = search_wikidata_property(token.text)
                    if wikidata_prop:
                        relation["wikidata_property"] = wikidata_prop

                    relationships.append(relation)

    return relationships

def annotate_text(text):
    """Annotate text with Wikidata entities and relationships."""
    entities = extract_entities(text)
    relationships = extract_relationships(text)
    
    entity_annotations = []
    for entity in entities:
        result = search_wikidata(entity)
        if result and result["relevance_score"] > 0.7:  # Set a threshold for filtering
            entity_annotations.append(result)

    output = {
        "original_sentence": text,
        "entities": entity_annotations,
        "relationships": relationships
    }

    return json.dumps(output, indent=4)  # Convert to JSON format

# Example Usage
if __name__ == "__main__":
    sentence = "Sigmund Freud developed psychoanalysis and influenced Carl Jung."
    json_output = annotate_text(sentence)
    print(json_output)


{
    "original_sentence": "Sigmund Freud developed psychoanalysis and influenced Carl Jung.",
    "entities": [
        {
            "entity": "Carl Jung",
            "wikidata_id": "Q41532",
            "label": "Carl Jung",
            "description": "Swiss psychiatrist and psychotherapist (1875\u20131961)",
            "wikidata_url": "https://www.wikidata.org/wiki/Q41532",
            "relevance_score": 1.0
        },
        {
            "entity": "Sigmund Freud",
            "wikidata_id": "Q9215",
            "label": "Sigmund Freud",
            "description": "Austrian neurologist and founder of psychoanalysis (1856\u20131939)",
            "wikidata_url": "https://www.wikidata.org/wiki/Q9215",
            "relevance_score": 1.0
        },
        {
            "entity": "psychoanalysis",
            "wikidata_id": "Q41630",
            "label": "psychoanalysis",
            "description": "psychological theory that was founded in 1890 by the Viennese neurologist Sigmund F

In [6]:


# Load sentences from the dataset
with open('/home/matt/Proj/Hermeticav2/testing/AnnotationTesting/TestSentancesGeneral.txt', "r", encoding="utf-8") as f:
    sentences = [line.strip() for line in f.readlines()]

# Annotate all sentences
results = []
for sentence in sentences:
    try:
        annotation = annotate_text(sentence)
        results.append(json.loads(annotation))  # Ensure JSON format
    except Exception as e:
        print(f"Error processing sentence: {sentence}")
        print(f"Exception: {e}")

# Save to a JSON file
with open('relationsannotation.json', "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print(f"Annotation completed. Results saved in {output_path}")


NameError: name 'output_path' is not defined

In [13]:
import spacy
import requests
import json
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer

# Load spaCy's English model and a sentence embedding model
nlp = spacy.load("en_core_web_lg")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

def string_similarity(a, b):
    """Compute string similarity score using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def sentence_similarity(sentence, entity_description):
    """Compute semantic similarity between the sentence and entity description."""
    sentence_embedding = embedding_model.encode(sentence)
    entity_embedding = embedding_model.encode(entity_description)
    
    # Compute cosine similarity
    similarity = sentence_embedding @ entity_embedding / (sum(sentence_embedding**2)**0.5 * sum(entity_embedding**2)**0.5)
    return similarity

def search_wikidata(entity, sentence):
    """Search Wikidata for the best entity match using sentence relevance scoring."""
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }

    response = requests.get(WIKIDATA_API_URL, params=params)
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            results = []
            
            for result in data["search"]:
                label = result.get("label", "")
                description = result.get("description", "")
                wikidata_id = result["id"]
                
                # Compute multiple relevance scores
                similarity = string_similarity(entity, label)
                sentence_sim = sentence_similarity(sentence, description) if description else 0
                length_score = len(description) / 100 if len(description) > 10 else 0  # Favor well-described entities
                
                # Final score: combination of all factors
                final_score = similarity + sentence_sim + length_score
                
                results.append({
                    "entity": entity,
                    "wikidata_id": wikidata_id,
                    "label": label,
                    "description": description,
                    "wikidata_url": f"https://www.wikidata.org/wiki/{wikidata_id}",
                    "relevance_score": round(final_score, 3)
                })

            # Pick the highest-ranked match
            if results:
                best_match = max(results, key=lambda x: x["relevance_score"])
                return best_match
    
    return None

def extract_entities(text):
    """Extract entities using improved multi-word detection."""
    doc = nlp(text)
    entities = set()

    # Extract named entities
    for ent in doc.ents:
        entities.add(ent.text)

    # Extract noun phrases (e.g., "theory of relativity")
    for chunk in doc.noun_chunks:
        clean_chunk = chunk.text.strip()
        if clean_chunk and len(clean_chunk) > 2:
            entities.add(clean_chunk)

    # Try longer multi-word entities first
    sorted_entities = sorted(entities, key=lambda x: -len(x.split()))

    final_entities = []
    seen_words = set()

    for entity in sorted_entities:
        if any(word in seen_words for word in entity.split()):
            continue  # Skip if words have already been added as a longer phrase
        
        final_entities.append(entity)
        seen_words.update(entity.split())

    return final_entities

def annotate_text(text):
    """Annotate text with Wikidata entities using sentence-based relevance scoring."""
    extracted_entities = extract_entities(text)

    entity_annotations = []
    for entity in extracted_entities:
        result = search_wikidata(entity, text)  # Use full sentence for scoring
        if result and result["relevance_score"] > 1.0:  # Ensure high confidence
            entity_annotations.append(result)

    output = {
        "original_sentence": text,
        "entities": entity_annotations
    }

    return json.dumps(output, indent=4)

# Example Usage
if __name__ == "__main__":
    sentence = "Albert Einstein developed the theory of relativity."
    json_output = annotate_text(sentence)
    print(json_output)


/home/matt/miniforge3/envs/Hermetica/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


{
    "original_sentence": "Albert Einstein developed the theory of relativity.",
    "entities": [
        {
            "entity": "Albert Einstein",
            "wikidata_id": "Q937",
            "label": "Albert Einstein",
            "description": "German-born theoretical physicist (1879\u20131955)",
            "wikidata_url": "https://www.wikidata.org/wiki/Q937",
            "relevance_score": 1.982
        },
        {
            "entity": "the theory",
            "wikidata_id": "Q27877266",
            "label": "The Theory",
            "description": "painting by Elliot Collins",
            "wikidata_url": "https://www.wikidata.org/wiki/Q27877266",
            "relevance_score": 1.314
        },
        {
            "entity": "relativity",
            "wikidata_id": "Q983751",
            "label": "relativity",
            "description": "quality of a property or measure that is defined in relation to some other entity, as opposed to absoluteness",
            "wikidata_u

In [14]:
import requests
import json

WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

def search_wikidata_debug(entity):
    """Search Wikidata and print all results for debugging."""
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }

    response = requests.get(WIKIDATA_API_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            print(f"\n🔍 **Results for entity: '{entity}'**")
            for result in data["search"]:
                print(f"📌 Wikidata ID: {result['id']}")
                print(f"   Label: {result.get('label', 'N/A')}")
                print(f"   Description: {result.get('description', 'N/A')}")
                print(f"   Wikidata URL: https://www.wikidata.org/wiki/{result['id']}\n")
        else:
            print(f"\n⚠️ No results found for '{entity}'\n")
    else:
        print(f"\n❌ API Error for '{entity}': {response.status_code}\n")

# Test entities
test_entities = ["Albert Einstein", "the theory", "relativity", "Tesla", "Amazon", "Python"]

# Run debug search on each entity
for entity in test_entities:
    search_wikidata_debug(entity)



🔍 **Results for entity: 'Albert Einstein'**
📌 Wikidata ID: Q937
   Label: Albert Einstein
   Description: German-born theoretical physicist (1879–1955)
   Wikidata URL: https://www.wikidata.org/wiki/Q937

📌 Wikidata ID: Q2030894
   Label: Albert Einstein College of Medicine
   Description: private medical school in New York City, NY
   Wikidata URL: https://www.wikidata.org/wiki/Q2030894

📌 Wikidata ID: Q21200226
   Label: Albert Einstein
   Description: Wikimedia permanent duplicate item
   Wikidata URL: https://www.wikidata.org/wiki/Q21200226

📌 Wikidata ID: Q47513150
   Label: Albert Einstein
   Description: painting by Max Westfield
   Wikidata URL: https://www.wikidata.org/wiki/Q47513150

📌 Wikidata ID: Q47510526
   Label: Albert Einstein
   Description: painting by Josef Scharl
   Wikidata URL: https://www.wikidata.org/wiki/Q47510526

📌 Wikidata ID: Q1630439
   Label: Albert Einstein Israelite Hospital
   Description: hospital in São Paulo, Brazil
   Wikidata URL: https://www.wi

In [15]:
import requests
import json
from difflib import SequenceMatcher

WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

def string_similarity(a, b):
    """Compute string similarity score using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_wikidata_filtered(entity, full_sentence):
    """Search Wikidata and rank results based on relevance to the full sentence."""
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }

    response = requests.get(WIKIDATA_API_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            results = []

            print(f"\n🔍 **Filtered results for entity: '{entity}'**")

            for result in data["search"]:
                label = result.get("label", "")
                description = result.get("description", "")
                wikidata_id = result["id"]

                # Compute multiple relevance scores
                similarity = string_similarity(entity, label)
                sentence_match = string_similarity(full_sentence, description) if description else 0
                
                # **Filter out non-relevant types** (paintings, books, TV episodes, etc.)
                if any(x in description.lower() for x in ["painting", "episode", "film", "song", "book"]):
                    continue  # Skip irrelevant entities
                
                # Final relevance score: boost if it matches both the entity and the sentence meaning
                final_score = similarity + (sentence_match * 2)  # Prioritize concepts matching the full sentence
                
                results.append({
                    "entity": entity,
                    "wikidata_id": wikidata_id,
                    "label": label,
                    "description": description,
                    "wikidata_url": f"https://www.wikidata.org/wiki/{wikidata_id}",
                    "relevance_score": round(final_score, 3)
                })
            
            # Sort by highest relevance
            results = sorted(results, key=lambda x: x["relevance_score"], reverse=True)

            for res in results:
                print(f"📌 {res['wikidata_id']} | {res['label']} | {res['description']} | Score: {res['relevance_score']}")

            # Return the best result
            return results[0] if results else None
        else:
            print(f"\n⚠️ No results found for '{entity}'\n")
    else:
        print(f"\n❌ API Error for '{entity}': {response.status_code}\n")

    return None

# Test sentence
sentence = "Albert Einstein developed the theory of relativity."

# Test entities
test_entities = ["Albert Einstein", "the theory", "relativity", "Tesla", "Amazon", "Python"]

# Run debug search on each entity
for entity in test_entities:
    search_wikidata_filtered(entity, sentence)



🔍 **Filtered results for entity: 'Albert Einstein'**
📌 Q937 | Albert Einstein | German-born theoretical physicist (1879–1955) | Score: 1.625
📌 Q21200226 | Albert Einstein | Wikimedia permanent duplicate item | Score: 1.518
📌 Q2030894 | Albert Einstein College of Medicine | private medical school in New York City, NY | Score: 1.238
📌 Q1630439 | Albert Einstein Israelite Hospital | hospital in São Paulo, Brazil | Score: 1.162

🔍 **Filtered results for entity: 'the theory'**
📌 Q114701924 | The theory of spanning trees |  | Score: 0.526
📌 Q114701898 | The theory of Archimedean bodies |  | Score: 0.476
📌 Q114701922 | The theory of resolvable field extensions |  | Score: 0.392
📌 Q114501447 | The theory of digit representation for real numbers |  | Score: 0.328

🔍 **Filtered results for entity: 'relativity'**
📌 Q114571573 | Relativity | written work by Albert Einstein | Score: 1.732
📌 Q18615221 | Relativity | 1233rd strip of the webcomic xkcd | Score: 1.714
📌 Q983751 | relativity | quality o

In [18]:
import spacy
import requests
import json
from difflib import SequenceMatcher

# Load spaCy's English model
nlp = spacy.load("en_core_web_lg")

WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

def string_similarity(a, b):
    """Compute string similarity score using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def extract_concepts(text):
    """Extract multi-word scientific/technical concepts using dependency parsing."""
    doc = nlp(text)
    concepts = set()

    # Extract named entities (e.g., "Albert Einstein")
    for ent in doc.ents:
        concepts.add(ent.text)

    # Extract noun phrases (e.g., "theory of relativity")
    for chunk in doc.noun_chunks:
        concepts.add(chunk.text.strip())

    # 🔍 NEW: Merge words that belong together (e.g., "Theory of Relativity")
    merged_concepts = set()
    for token in doc:
        # Merge if an adjective modifies a noun (e.g., "Relativity" modifies "Theory")
        if token.dep_ == "amod" and token.head.pos_ == "NOUN":
            merged_concepts.add(f"{token.text} {token.head.text}")

    # Combine merged concepts with original ones
    concepts.update(merged_concepts)

    # Convert to sorted list (longest phrases first)
    sorted_concepts = sorted(concepts, key=lambda x: -len(x.split()))
    return sorted_concepts

def search_wikidata_filtered(entity, full_sentence):
    """Search Wikidata and rank results based on better sentence relevance matching."""
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }

    response = requests.get(WIKIDATA_API_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            results = []
            sentence_keywords = extract_concepts(full_sentence)

            print(f"\n🔍 **Filtered results for entity: '{entity}'**")

            for result in data["search"]:
                label = result.get("label", "")
                description = result.get("description", "")
                wikidata_id = result["id"]

                # Compute multiple relevance scores
                similarity = string_similarity(entity, label)
                sentence_match = string_similarity(full_sentence, description) if description else 0
                
                # **Filter out non-relevant types** (paintings, books, TV episodes, etc.)
                if any(x in description.lower() for x in ["painting", "episode", "film", "song", "book"]):
                    continue  # Skip irrelevant entities
                
                # **Boost relevance if description contains important words from the sentence**
                keyword_match_score = sum(1 for word in sentence_keywords if word in description.lower()) * 0.5  # Extra weight
                
                # Final score: combination of similarity, sentence meaning, and keyword matching
                final_score = similarity + (sentence_match * 2) + keyword_match_score
                
                results.append({
                    "entity": entity,
                    "wikidata_id": wikidata_id,
                    "label": label,
                    "description": description,
                    "wikidata_url": f"https://www.wikidata.org/wiki/{wikidata_id}",
                    "relevance_score": round(final_score, 3)
                })
            
            # Sort by highest relevance
            results = sorted(results, key=lambda x: x["relevance_score"], reverse=True)

            for res in results:
                print(f"📌 {res['wikidata_id']} | {res['label']} | {res['description']} | Score: {res['relevance_score']}")

            # Return the best result
            return results[0] if results else None
        else:
            print(f"\n⚠️ No results found for '{entity}'\n")
    else:
        print(f"\n❌ API Error for '{entity}': {response.status_code}\n")

    return None

# Test sentence
sentence = "Albert Einstein developed the theory of relativity."

# Extract **merged** concepts before searching
merged_concepts = extract_concepts(sentence)
print("\n🔍 **Merged Concepts Detected:**", merged_concepts)

# Run Wikidata search for each merged concept
for concept in merged_concepts:
    search_wikidata_filtered(concept, sentence)



🔍 **Merged Concepts Detected:** ['Albert Einstein', 'the theory', 'relativity']

🔍 **Filtered results for entity: 'Albert Einstein'**
📌 Q937 | Albert Einstein | German-born theoretical physicist (1879–1955) | Score: 1.625
📌 Q21200226 | Albert Einstein | Wikimedia permanent duplicate item | Score: 1.518
📌 Q2030894 | Albert Einstein College of Medicine | private medical school in New York City, NY | Score: 1.238
📌 Q1630439 | Albert Einstein Israelite Hospital | hospital in São Paulo, Brazil | Score: 1.162

🔍 **Filtered results for entity: 'the theory'**
📌 Q114701924 | The theory of spanning trees |  | Score: 0.526
📌 Q114701898 | The theory of Archimedean bodies |  | Score: 0.476
📌 Q114701922 | The theory of resolvable field extensions |  | Score: 0.392
📌 Q114501447 | The theory of digit representation for real numbers |  | Score: 0.328

🔍 **Filtered results for entity: 'relativity'**
📌 Q114571573 | Relativity | written work by Albert Einstein | Score: 1.732
📌 Q18615221 | Relativity | 12

In [21]:
import spacy
import requests
import json
from difflib import SequenceMatcher

# Load spaCy's English model
nlp = spacy.load("en_core_web_lg")

WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

def string_similarity(a, b):
    """Compute string similarity score using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_wikidata(entity):
    """Search Wikidata for an entity and return all possible matches."""
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }
    response = requests.get(WIKIDATA_API_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            return [{
                "entity": entity,
                "wikidata_id": result["id"],
                "label": result.get("label", ""),
                "description": result.get("description", ""),
                "wikidata_url": f"https://www.wikidata.org/wiki/{result['id']}"
            } for result in data["search"]]
    return []

def get_wikidata_connections(entity_id):
    """Retrieve all entities directly connected to a Wikidata entity."""
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        connections = set()
        
        # Debugging: Print raw JSON to check structure
        print(f"\n📡 Wikidata response for {entity_id}: {json.dumps(data, indent=2)[:1000]}...\n")

        # Navigate JSON structure
        entity_data = data.get("entities", {}).get(entity_id, {})
        claims = entity_data.get("claims", {})

        # Extract linked entities
        for prop, values in claims.items():
            for value in values:
                if "mainsnak" in value and "datavalue" in value["mainsnak"]:
                    val_data = value["mainsnak"]["datavalue"]["value"]
                    if isinstance(val_data, dict) and "id" in val_data:
                        connections.add(val_data["id"])  # Store linked entity ID
                        
        return connections
    return set()

def compute_relevance_score(entity_id, sentence_entities):
    """Compute a relevance score based on connectivity to other entities in the sentence."""
    connections = get_wikidata_connections(entity_id)
    if not connections:
        return 0  # If no links, it's likely not relevant
    
    # Debugging: Print how many entities are linked
    print(f"🔗 Entity {entity_id} is connected to {len(connections)} other Wikidata entities.")

    # Count how many sentence entities are linked to this one
    relevance = sum(1 for e in sentence_entities if e["wikidata_id"] in connections)
    
    return relevance

def extract_entities(text):
    """Extract key entities from a sentence using spaCy."""
    doc = nlp(text)
    entities = set()

    # Extract proper named entities
    for ent in doc.ents:
        entities.add(ent.text)

    # Extract noun phrases (e.g., "theory of relativity")
    for chunk in doc.noun_chunks:
        entities.add(chunk.text.strip())

    # **Force-merge key scientific concepts**
    merged = set()
    for token in doc:
        if token.dep_ == "amod" and token.head.pos_ == "NOUN":
            merged.add(f"{token.text} {token.head.text}")

    # **Explicitly include "Theory of Relativity" if found**
    if "theory of relativity" in text.lower():
        merged.add("Theory of Relativity")

    entities.update(merged)

    return list(entities)

def annotate_text(text):
    """Annotate text with Wikidata entities using graph-based relevance scoring."""
    extracted_entities = extract_entities(text)
    
    entity_candidates = {}
    for entity in extracted_entities:
        candidates = search_wikidata(entity)
        entity_candidates[entity] = candidates

    # Compute relevance scores for all candidates
    best_entities = []
    for entity, candidates in entity_candidates.items():
        if not candidates:
            continue
        
        # Compute connectivity-based relevance score
        for candidate in candidates:
            candidate["relevance_score"] = compute_relevance_score(candidate["wikidata_id"], candidates)
        
        # Pick the best match
        best_match = max(candidates, key=lambda x: x["relevance_score"], default=None)
        if best_match and best_match["relevance_score"] > 0:
            best_entities.append(best_match)

    output = {
        "original_sentence": text,
        "entities": best_entities
    }

    return json.dumps(output, indent=4)

# Example Usage
if __name__ == "__main__":
    sentence = "Albert Einstein developed the theory of relativity."
    json_output = annotate_text(sentence)
    print(json_output)



📡 Wikidata response for Q937: {
  "entities": {
    "Q937": {
      "pageid": 1262,
      "ns": 0,
      "title": "Q937",
      "lastrevid": 2313896021,
      "modified": "2025-02-19T15:35:09Z",
      "type": "item",
      "id": "Q937",
      "labels": {
        "ab": {
          "language": "ab",
          "value": "\u0410\u043b\u0431\u0435\u0440\u0442 \u0415\u0438\u043d\u0448\u0442\u0435\u0438\u043d"
        },
        "ace": {
          "language": "ace",
          "value": "Albert Einstein"
        },
        "aeb-arab": {
          "language": "aeb-arab",
          "value": "\u0623\u0644\u0628\u0627\u0631\u062a \u0625\u0646\u0634\u062a\u0627\u064a\u0646"
        },
        "af": {
          "language": "af",
          "value": "Albert Einstein"
        },
        "am": {
          "language": "am",
          "value": "\u12a0\u120d\u1260\u122d\u1275 \u12a0\u12ed\u1295\u1235\u1273\u12ed\u1295"
        },
        "an": {
          "language": "an",
          "value": "Albert Einstei

In [22]:
import spacy
import requests
import json
import networkx as nx
from collections import defaultdict
from difflib import SequenceMatcher

nlp = spacy.load("en_core_web_lg")
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

def string_similarity(a, b):
    """Compute similarity score using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_wikidata(entity):
    """Search Wikidata for an entity and return all possible matches."""
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }
    response = requests.get(WIKIDATA_API_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            return [{
                "entity": entity,
                "wikidata_id": result["id"],
                "label": result.get("label", ""),
                "description": result.get("description", ""),
                "wikidata_url": f"https://www.wikidata.org/wiki/{result['id']}"
            } for result in data["search"]]
    return []

def get_wikidata_connections(entity_id):
    """Retrieve entities directly linked to a Wikidata entity."""
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        connections = set()

        entity_data = data.get("entities", {}).get(entity_id, {})
        claims = entity_data.get("claims", {})

        for prop, values in claims.items():
            for value in values:
                if "mainsnak" in value and "datavalue" in value["mainsnak"]:
                    val_data = value["mainsnak"]["datavalue"]["value"]
                    if isinstance(val_data, dict) and "id" in val_data:
                        connections.add(val_data["id"])

        return connections
    return set()

def extract_entities(text):
    """Extract key entities from a sentence using spaCy."""
    doc = nlp(text)
    entities = set()

    for ent in doc.ents:
        entities.add(ent.text)

    for chunk in doc.noun_chunks:
        entities.add(chunk.text.strip())

    merged = set()
    for token in doc:
        if token.dep_ == "amod" and token.head.pos_ == "NOUN":
            merged.add(f"{token.text} {token.head.text}")

    entities.update(merged)
    return list(entities)

def build_wikidata_graph(entity_candidates):
    """Construct a graph where nodes are Wikidata entities and edges are relationships."""
    graph = nx.Graph()
    entity_id_map = {}

    # Add entities and their direct connections
    for entity, candidates in entity_candidates.items():
        for candidate in candidates:
            entity_id = candidate["wikidata_id"]
            graph.add_node(entity_id, label=candidate["label"])
            entity_id_map[entity] = entity_id
            
            connections = get_wikidata_connections(entity_id)
            for linked_entity in connections:
                graph.add_edge(entity_id, linked_entity)

    return graph, entity_id_map

def rank_entities(graph, entity_id_map):
    """Rank entities based on their centrality in the Wikidata graph."""
    if len(graph.nodes) == 0:
        return {}

    centrality = nx.pagerank(graph)
    ranked_entities = {
        entity: centrality.get(entity_id_map[entity], 0)
        for entity in entity_id_map
    }
    return ranked_entities

def annotate_text(text):
    """Annotate text with Wikidata entities using graph-based ranking."""
    extracted_entities = extract_entities(text)

    # Step 1: Search Wikidata
    entity_candidates = {entity: search_wikidata(entity) for entity in extracted_entities}

    # Step 2: Build knowledge graph
    graph, entity_id_map = build_wikidata_graph(entity_candidates)

    # Step 3: Rank entities based on connectivity
    entity_ranks = rank_entities(graph, entity_id_map)

    # Step 4: Select the best match for each entity
    best_entities = []
    for entity, candidates in entity_candidates.items():
        if not candidates:
            continue

        for candidate in candidates:
            candidate["relevance_score"] = entity_ranks.get(entity, 0)

        best_match = max(candidates, key=lambda x: x["relevance_score"], default=None)
        if best_match and best_match["relevance_score"] > 0:
            best_entities.append(best_match)

    output = {
        "original_sentence": text,
        "entities": best_entities
    }

    return json.dumps(output, indent=4)

# **Test Case**
if __name__ == "__main__":
    sentence = "Albert Einstein developed the theory of relativity."
    json_output = annotate_text(sentence)
    print(json_output)


{
    "original_sentence": "Albert Einstein developed the theory of relativity.",
    "entities": [
        {
            "entity": "Albert Einstein",
            "wikidata_id": "Q937",
            "label": "Albert Einstein",
            "description": "German-born theoretical physicist (1879\u20131955)",
            "wikidata_url": "https://www.wikidata.org/wiki/Q937",
            "relevance_score": 0.004725876972024288
        },
        {
            "entity": "the theory",
            "wikidata_id": "Q15079318",
            "label": "The Theory of Everything",
            "description": "2014 film directed by James Marsh",
            "wikidata_url": "https://www.wikidata.org/wiki/Q15079318",
            "relevance_score": 0.005100216282728562
        },
        {
            "entity": "relativity",
            "wikidata_id": "Q983751",
            "label": "relativity",
            "description": "quality of a property or measure that is defined in relation to some other entity, a

In [23]:
import spacy
import requests
import json
import networkx as nx
from collections import defaultdict
from difflib import SequenceMatcher

# Load NLP model
nlp = spacy.load("en_core_web_lg")

# Wikidata API
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

### ======= Utility Functions ======= ###

def string_similarity(a, b):
    """Compute string similarity between two words."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_wikipedia(entity):
    """Search Wikipedia/Wikidata for possible matches."""
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }
    response = requests.get(WIKIDATA_API_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if "search" in data:
            return [{
                "entity": entity,
                "wikidata_id": result["id"],
                "label": result.get("label", ""),
                "description": result.get("description", ""),
                "wikidata_url": f"https://www.wikidata.org/wiki/{result['id']}"
            } for result in data["search"]]
    return []

def get_wikidata_connections(entity_id):
    """Retrieve entities directly linked to a Wikidata entity."""
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        connections = set()

        entity_data = data.get("entities", {}).get(entity_id, {})
        claims = entity_data.get("claims", {})

        for prop, values in claims.items():
            for value in values:
                if "mainsnak" in value and "datavalue" in value["mainsnak"]:
                    val_data = value["mainsnak"]["datavalue"]["value"]
                    if isinstance(val_data, dict) and "id" in val_data:
                        connections.add(val_data["id"])

        return connections
    return set()

### ======= Entity Extraction ======= ###

def extract_entities(text):
    """Extract possible entities from a sentence using NLP."""
    doc = nlp(text)
    entities = set()

    for ent in doc.ents:
        entities.add(ent.text)

    for chunk in doc.noun_chunks:
        entities.add(chunk.text.strip())

    return list(entities)

def build_wikidata_graph(entity_candidates):
    """Construct a graph where nodes are Wikidata entities and edges are relationships."""
    graph = nx.Graph()
    entity_id_map = {}

    # Add entities and their direct connections
    for entity, candidates in entity_candidates.items():
        for candidate in candidates:
            entity_id = candidate["wikidata_id"]
            graph.add_node(entity_id, label=candidate["label"])
            entity_id_map[entity] = entity_id
            
            connections = get_wikidata_connections(entity_id)
            for linked_entity in connections:
                graph.add_edge(entity_id, linked_entity)

    return graph, entity_id_map

def rank_entities(graph, entity_id_map):
    """Rank entities based on their centrality in the Wikidata graph."""
    if len(graph.nodes) == 0:
        return {}

    centrality = nx.pagerank(graph)
    ranked_entities = {
        entity: centrality.get(entity_id_map[entity], 0)
        for entity in entity_id_map
    }
    return ranked_entities

def refine_entities_with_graph(entity_candidates):
    """Refine entities by checking how well they are connected in Wikidata."""
    graph, entity_id_map = build_wikidata_graph(entity_candidates)
    entity_ranks = rank_entities(graph, entity_id_map)

    best_entities = []
    for entity, candidates in entity_candidates.items():
        if not candidates:
            continue

        for candidate in candidates:
            candidate["relevance_score"] = entity_ranks.get(entity, 0)

        best_match = max(candidates, key=lambda x: x["relevance_score"], default=None)
        if best_match and best_match["relevance_score"] > 0:
            best_entities.append(best_match)

    return best_entities

### ======= Main Function ======= ###

def annotate_text(text):
    """Annotate text with Wikipedia/Wikidata entities using graph-based correction."""
    extracted_entities = extract_entities(text)

    # Step 1: Search Wikipedia/Wikidata
    entity_candidates = {entity: search_wikipedia(entity) for entity in extracted_entities}

    # Step 2: Refine entity selection using Wikidata relationships
    refined_entities = refine_entities_with_graph(entity_candidates)

    output = {
        "original_sentence": text,
        "entities": refined_entities
    }

    return json.dumps(output, indent=4)

# **Test Case**
if __name__ == "__main__":
    sentence = "Albert Einstein developed the theory of relativity."
    json_output = annotate_text(sentence)
    print(json_output)


{
    "original_sentence": "Albert Einstein developed the theory of relativity.",
    "entities": [
        {
            "entity": "Albert Einstein",
            "wikidata_id": "Q937",
            "label": "Albert Einstein",
            "description": "German-born theoretical physicist (1879\u20131955)",
            "wikidata_url": "https://www.wikidata.org/wiki/Q937",
            "relevance_score": 0.004725876972024288
        },
        {
            "entity": "the theory",
            "wikidata_id": "Q15079318",
            "label": "The Theory of Everything",
            "description": "2014 film directed by James Marsh",
            "wikidata_url": "https://www.wikidata.org/wiki/Q15079318",
            "relevance_score": 0.005100216282728562
        },
        {
            "entity": "relativity",
            "wikidata_id": "Q983751",
            "label": "relativity",
            "description": "quality of a property or measure that is defined in relation to some other entity, a

In [24]:
import spacy
import requests
import json
from collections import defaultdict

# Load NLP model
nlp = spacy.load("en_core_web_lg")

# Wikidata API
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

### ======= Utility Functions ======= ###

def search_wikipedia(entity):
    """Search Wikipedia/Wikidata for possible matches."""
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }
    response = requests.get(WIKIDATA_API_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if "search" in data:
            return [{
                "entity": entity,
                "wikidata_id": result["id"],
                "label": result.get("label", ""),
                "description": result.get("description", ""),
                "wikidata_url": f"https://www.wikidata.org/wiki/{result['id']}"
            } for result in data["search"]]
    return []

def get_direct_wikidata_connections(entity_id):
    """Retrieve directly connected entities from Wikidata."""
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        connections = set()

        entity_data = data.get("entities", {}).get(entity_id, {})
        claims = entity_data.get("claims", {})

        for prop, values in claims.items():
            for value in values:
                if "mainsnak" in value and "datavalue" in value["mainsnak"]:
                    val_data = value["mainsnak"]["datavalue"]["value"]
                    if isinstance(val_data, dict) and "id" in val_data:
                        connections.add(val_data["id"])

        return connections
    return set()

def extract_entities(text):
    """Extract possible entities from a sentence using NLP."""
    doc = nlp(text)
    entities = set()

    for ent in doc.ents:
        entities.add(ent.text)

    for chunk in doc.noun_chunks:
        entities.add(chunk.text.strip())

    return list(entities)

def rank_entities_by_connections(entity_candidates):
    """Rank entities by the number of direct Wikidata connections."""
    entity_scores = {}

    for entity, candidates in entity_candidates.items():
        if not candidates:
            continue

        for candidate in candidates:
            candidate["connections"] = len(get_direct_wikidata_connections(candidate["wikidata_id"]))

        best_match = max(candidates, key=lambda x: x["connections"], default=None)
        if best_match:
            entity_scores[entity] = best_match

    return entity_scores

### ======= Main Function ======= ###

def annotate_text(text):
    """Annotate text with Wikipedia/Wikidata entities using direct connections."""
    extracted_entities = extract_entities(text)

    # Step 1: Search Wikipedia/Wikidata
    entity_candidates = {entity: search_wikipedia(entity) for entity in extracted_entities}

    # Step 2: Rank by direct connections
    ranked_entities = rank_entities_by_connections(entity_candidates)

    # Step 3: Convert to output format
    final_entities = []
    for entity, data in ranked_entities.items():
        final_entities.append({
            "entity": entity,
            "wikidata_id": data["wikidata_id"],
            "label": data["label"],
            "description": data["description"],
            "wikidata_url": data["wikidata_url"],
            "relevance_score": data["connections"]  # Direct connection count
        })

    output = {
        "original_sentence": text,
        "entities": final_entities
    }

    return json.dumps(output, indent=4)

# **Test Case**
if __name__ == "__main__":
    sentence = "Albert Einstein developed the theory of relativity."
    json_output = annotate_text(sentence)
    print(json_output)


{
    "original_sentence": "Albert Einstein developed the theory of relativity.",
    "entities": [
        {
            "entity": "Albert Einstein",
            "wikidata_id": "Q937",
            "label": "Albert Einstein",
            "description": "German-born theoretical physicist (1879\u20131955)",
            "wikidata_url": "https://www.wikidata.org/wiki/Q937",
            "relevance_score": 184
        },
        {
            "entity": "the theory",
            "wikidata_id": "Q15079318",
            "label": "The Theory of Everything",
            "description": "2014 film directed by James Marsh",
            "wikidata_url": "https://www.wikidata.org/wiki/Q15079318",
            "relevance_score": 86
        },
        {
            "entity": "relativity",
            "wikidata_id": "Q8010281",
            "label": "Relativity",
            "description": "episode of Star Trek: Voyager (S5 E24)",
            "wikidata_url": "https://www.wikidata.org/wiki/Q8010281",
       

In [25]:
import spacy
import requests
import json
from collections import defaultdict

# Load NLP model
nlp = spacy.load("en_core_web_lg")

# Wikidata API
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

# Entity types to prioritize
RELEVANT_ENTITY_TYPES = {"human", "scientific concept", "theory", "book", "scientific work"}

### ======= Utility Functions ======= ###

def search_wikipedia(entity):
    """Search Wikipedia/Wikidata for possible matches."""
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }
    response = requests.get(WIKIDATA_API_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if "search" in data:
            return [{
                "entity": entity,
                "wikidata_id": result["id"],
                "label": result.get("label", ""),
                "description": result.get("description", ""),
                "wikidata_url": f"https://www.wikidata.org/wiki/{result['id']}"
            } for result in data["search"]]
    return []

def get_direct_wikidata_connections(entity_id):
    """Retrieve directly connected entities from Wikidata."""
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        connections = set()

        entity_data = data.get("entities", {}).get(entity_id, {})
        claims = entity_data.get("claims", {})

        for prop, values in claims.items():
            for value in values:
                if "mainsnak" in value and "datavalue" in value["mainsnak"]:
                    val_data = value["mainsnak"]["datavalue"]["value"]
                    if isinstance(val_data, dict) and "id" in val_data:
                        connections.add(val_data["id"])

        return connections
    return set()

def filter_relevant_entities(candidates):
    """Filter out irrelevant entities based on description keywords."""
    filtered = []
    for entity in candidates:
        desc = entity.get("description", "").lower()
        if any(keyword in desc for keyword in RELEVANT_ENTITY_TYPES):
            filtered.append(entity)
    return filtered if filtered else candidates  # If nothing is relevant, keep original set

def extract_entities(text):
    """Extract possible entities from a sentence using NLP."""
    doc = nlp(text)
    entities = set()

    for ent in doc.ents:
        entities.add(ent.text)

    for chunk in doc.noun_chunks:
        entities.add(chunk.text.strip())

    return list(entities)

def rank_entities_by_relevance(entity_candidates):
    """Rank entities based on number of direct Wikidata connections and similarity to extracted terms."""
    entity_scores = {}

    for entity, candidates in entity_candidates.items():
        if not candidates:
            continue

        filtered_candidates = filter_relevant_entities(candidates)

        for candidate in filtered_candidates:
            candidate["connections"] = len(get_direct_wikidata_connections(candidate["wikidata_id"]))

        best_match = max(filtered_candidates, key=lambda x: x["connections"], default=None)
        if best_match:
            entity_scores[entity] = best_match

    return entity_scores

### ======= Main Function ======= ###

def annotate_text(text):
    """Annotate text with Wikipedia/Wikidata entities using direct connections & semantic filtering."""
    extracted_entities = extract_entities(text)

    # Step 1: Search Wikipedia/Wikidata
    entity_candidates = {entity: search_wikipedia(entity) for entity in extracted_entities}

    # Step 2: Rank by direct connections & semantic relevance
    ranked_entities = rank_entities_by_relevance(entity_candidates)

    # Step 3: Convert to output format
    final_entities = []
    for entity, data in ranked_entities.items():
        final_entities.append({
            "entity": entity,
            "wikidata_id": data["wikidata_id"],
            "label": data["label"],
            "description": data["description"],
            "wikidata_url": data["wikidata_url"],
            "relevance_score": data["connections"]  # Direct connection count
        })

    output = {
        "original_sentence": text,
        "entities": final_entities
    }

    return json.dumps(output, indent=4)

# **Test Case**
if __name__ == "__main__":
    sentence = "Albert Einstein developed the theory of relativity."
    json_output = annotate_text(sentence)
    print(json_output)


{
    "original_sentence": "Albert Einstein developed the theory of relativity.",
    "entities": [
        {
            "entity": "Albert Einstein",
            "wikidata_id": "Q937",
            "label": "Albert Einstein",
            "description": "German-born theoretical physicist (1879\u20131955)",
            "wikidata_url": "https://www.wikidata.org/wiki/Q937",
            "relevance_score": 184
        },
        {
            "entity": "the theory",
            "wikidata_id": "Q23012191",
            "label": "The Theory of Economic Development",
            "description": "book by Joseph Schumpeter",
            "wikidata_url": "https://www.wikidata.org/wiki/Q23012191",
            "relevance_score": 4
        },
        {
            "entity": "relativity",
            "wikidata_id": "Q8010281",
            "label": "Relativity",
            "description": "episode of Star Trek: Voyager (S5 E24)",
            "wikidata_url": "https://www.wikidata.org/wiki/Q8010281",
      

In [28]:
import spacy
import requests
import re
import json
from difflib import SequenceMatcher

# Load spaCy's English model
nlp = spacy.load("en_core_web_lg")

def string_similarity(a, b):
    """Compute string similarity score using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_wikidata(entity, entity_type="item"):
    """Search Wikidata for an entity/property and return the best match."""
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity,
        "type": entity_type  # Search for "item" (entity) or "property"
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            best_match = None
            highest_score = 0.0
            
            for result in data["search"]:
                label = result.get("label", "")
                description = result.get("description", "")
                wikidata_id = result["id"]

                # Compute similarity score
                similarity = string_similarity(entity, label)

                # Final score
                final_score = similarity
                
                if final_score > highest_score:
                    highest_score = final_score
                    best_match = {
                        "entity": entity,
                        "wikidata_id": wikidata_id,
                        "label": label,
                        "description": description,
                        "wikidata_url": f"https://www.wikidata.org/wiki/{wikidata_id}",
                        "relevance_score": round(final_score, 3)
                    }
            
            return best_match  # Return only the best match
    return None

def clean_entity(entity):
    """Clean entity by removing stop words and special characters."""
    entity = entity.lower().strip()
    entity = re.sub(r'[^\w\s]', '', entity)  # Remove punctuation
    return entity

def extract_entities(text):
    """Extract entities using both NER and keyword extraction."""
    doc = nlp(text)
    entities = set(ent.text for ent in doc.ents)  # Extract named entities
    
    # Extract additional keywords (noun chunks)
    for chunk in doc.noun_chunks:
        clean_chunk = clean_entity(chunk.text)
        if clean_chunk and len(clean_chunk) > 2:  # Avoid short words
            entities.add(chunk.text)

    return list(entities)

def extract_relationships(doc):
    """Extract subject-predicate-object triples using dependency parsing."""
    relationships = []
    for token in doc:
        if token.pos_ == "VERB":
            subj = None
            obj = None
            # Find subject (nsubj or nsubjpass)
            subj_tokens = [child for child in token.children if child.dep_ in ("nsubj", "nsubjpass")]
            if subj_tokens:
                subj = ' '.join([t.text for t in subj_tokens[0].subtree])
            
            # Find object (dobj, attr, or prepositional object)
            obj_tokens = [child for child in token.children if child.dep_ in ("dobj", "attr", "prep")]
            if obj_tokens:
                obj = ' '.join([t.text for t in obj_tokens[0].subtree])
            
            if subj and obj:
                relationships.append({
                    "subject": subj,
                    "predicate": token.lemma_,  # Use lemma (e.g., "found" instead of "founded")
                    "object": obj
                })
    return relationships

def annotate_text(text):
    """Annotate text with entities and relationships."""
    doc = nlp(text)
    entities = extract_entities(text)
    
    # Annotate entities
    annotations = []
    for entity in entities:
        result = search_wikidata(entity)
        if result and result["relevance_score"] > 0.7:
            annotations.append(result)
    
    # Extract and annotate relationships
    relationships = []
    for rel in extract_relationships(doc):
        # Search for the predicate as a Wikidata property
        predicate_property = search_wikidata(rel["predicate"], entity_type="property")
        if predicate_property and predicate_property["relevance_score"] > 0.5:
            relationships.append({
                "subject": rel["subject"],
                "predicate": rel["predicate"],
                "object": rel["object"],
                "property_id": predicate_property["wikidata_id"],
                "property_label": predicate_property["label"]
            })
    
    output = {
        "original_sentence": text,
        "annotations": annotations,
        "relationships": relationships
    }
    return json.dumps(output, indent=4)

# Example Usage
if __name__ == "__main__":
    sentence = "Einstien developed the theory of relativity"
    json_output = annotate_text(sentence)
    print(json_output)

{
    "original_sentence": "Einstien developed the theory of relativity",
    "annotations": [
        {
            "entity": "the theory",
            "wikidata_id": "Q27877266",
            "label": "The Theory",
            "description": "painting by Elliot Collins",
            "wikidata_url": "https://www.wikidata.org/wiki/Q27877266",
            "relevance_score": 1.0
        },
        {
            "entity": "relativity",
            "wikidata_id": "Q983751",
            "label": "relativity",
            "description": "quality of a property or measure that is defined in relation to some other entity, as opposed to absoluteness",
            "wikidata_url": "https://www.wikidata.org/wiki/Q983751",
            "relevance_score": 1.0
        }
    ],
    "relationships": [
        {
            "subject": "Einstien",
            "predicate": "develop",
            "object": "the theory of relativity",
            "property_id": "P178",
            "property_label": "developer"

In [29]:


# Load sentences from the dataset
with open('/home/matt/Proj/Hermeticav2/testing/AnnotationTesting/TestSentancesGeneral.txt', "r", encoding="utf-8") as f:
    sentences = [line.strip() for line in f.readlines()]

# Annotate all sentences
results = []
for sentence in sentences:
    try:
        annotation = annotate_text(sentence)
        results.append(json.loads(annotation))  # Ensure JSON format
    except Exception as e:
        print(f"Error processing sentence: {sentence}")
        print(f"Exception: {e}")

# Save to a JSON file
with open('relationsannotation.json', "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print(f"Annotation completed.")


Annotation completed.


In [32]:
import spacy
import requests
import re
import json
from difflib import SequenceMatcher

# Load spaCy's English model
nlp = spacy.load("en_core_web_lg")

def string_similarity(a, b):
    """Compute string similarity score using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_wikidata(entity, entity_type="item"):
    """Search Wikidata for an entity/property and return the best match."""
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity,
        "type": entity_type  # Search for "item" (entity) or "property"
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            best_match = None
            highest_score = 0.0
            
            for result in data["search"]:
                label = result.get("label", "")
                description = result.get("description", "")
                wikidata_id = result["id"]

                # Compute similarity score
                similarity = string_similarity(entity, label)

                # Final score
                final_score = similarity
                
                if final_score > highest_score:
                    highest_score = final_score
                    best_match = {
                        "entity": entity,
                        "wikidata_id": wikidata_id,
                        "label": label,
                        "description": description,
                        "wikidata_url": f"https://www.wikidata.org/wiki/{wikidata_id}",
                        "relevance_score": round(final_score, 3)
                    }
            
            return best_match  # Return only the best match
    return None

def clean_entity(entity):
    """Clean entity by removing stop words and special characters."""
    entity = entity.lower().strip()
    entity = re.sub(r'[^\w\s]', '', entity)  # Remove punctuation
    return entity

def extract_entities(text):
    """Extract entities using both NER and keyword extraction."""
    doc = nlp(text)
    entities = set(ent.text for ent in doc.ents)  # Extract named entities
    
    # Extract additional keywords (noun chunks)
    for chunk in doc.noun_chunks:
        clean_chunk = clean_entity(chunk.text)
        if clean_chunk and len(clean_chunk) > 2:  # Avoid short words
            entities.add(chunk.text)

    return list(entities)
def extract_relationships(doc):
    for ent in doc.ents:
        if ent.label_ in ["DATE", "CARDINAL"]:
            relationships.append({
                "subject": subject_entity,
                "predicate": "has_value",
                "object": ent.text,
                "property_id": "P2067"  # molecular weight, etc.
            })
def extract_relationships(doc):
    """Extract subject-predicate-object triples using dependency parsing."""
    relationships = []
    for token in doc:
        if token.pos_ == "VERB":
            subj = None
            obj = None
            # Find subject (nsubj or nsubjpass)
            subj_tokens = [child for child in token.children if child.dep_ in ("nsubj", "nsubjpass")]
            if subj_tokens:
                subj = ' '.join([t.text for t in subj_tokens[0].subtree])
            
            # Find object (dobj, attr, or prepositional object)
            obj_tokens = [child for child in token.children if child.dep_ in ("dobj", "attr", "prep")]
            if obj_tokens:
                obj = ' '.join([t.text for t in obj_tokens[0].subtree])
            
            if subj and obj:
                relationships.append({
                    "subject": subj,
                    "predicate": token.lemma_,  # Use lemma (e.g., "found" instead of "founded")
                    "object": obj
                })
    return relationships

def annotate_text(text):
    """Annotate text with entities and relationships."""
    doc = nlp(text)
    entities = extract_entities(text)
    
    # Annotate entities
    annotations = []
    for entity in entities:
        result = search_wikidata(entity)
        if result and result["relevance_score"] > 0.7:
            annotations.append(result)
    
    # Extract and annotate relationships
    relationships = []
    for rel in extract_relationships(doc):
        # Search for the predicate as a Wikidata property
        predicate_property = search_wikidata(rel["predicate"], entity_type="property")
        if predicate_property and predicate_property["relevance_score"] > 0.5:
            relationships.append({
                "subject": rel["subject"],
                "predicate": rel["predicate"],
                "object": rel["object"],
                "property_id": predicate_property["wikidata_id"],
                "property_label": predicate_property["label"]
            })
    
    output = {
        "original_sentence": text,
        "annotations": annotations,
        "relationships": relationships
    }
    return json.dumps(output, indent=4)

# Example Usage
if __name__ == "__main__":
    sentence = "Albert Einstien developed the theory of relativity"
    json_output = annotate_text(sentence)
    print(json_output)

{
    "original_sentence": "Albert Einstien developed the theory of relativity",
    "annotations": [
        {
            "entity": "the theory",
            "wikidata_id": "Q27877266",
            "label": "The Theory",
            "description": "painting by Elliot Collins",
            "wikidata_url": "https://www.wikidata.org/wiki/Q27877266",
            "relevance_score": 1.0
        },
        {
            "entity": "relativity",
            "wikidata_id": "Q983751",
            "label": "relativity",
            "description": "quality of a property or measure that is defined in relation to some other entity, as opposed to absoluteness",
            "wikidata_url": "https://www.wikidata.org/wiki/Q983751",
            "relevance_score": 1.0
        }
    ],
    "relationships": [
        {
            "subject": "Albert Einstien",
            "predicate": "develop",
            "object": "the theory of relativity",
            "property_id": "P178",
            "property_label

In [33]:
import spacy
import requests
import re
import json
from difflib import SequenceMatcher

# Load spaCy English model
nlp = spacy.load("en_core_web_lg")

def string_similarity(a, b):
    """Compute similarity score between two strings."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_wikidata(entity, entity_type="item"):
    """Search Wikidata for an entity and return the best match."""
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity,
        "type": entity_type
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            best_match = None
            highest_score = 0.0
            
            for result in data["search"]:
                label = result.get("label", "")
                description = result.get("description", "")
                wikidata_id = result["id"]

                # Compute similarity score
                similarity = string_similarity(entity, label)

                # Final score
                final_score = similarity
                
                if final_score > highest_score:
                    highest_score = final_score
                    best_match = {
                        "entity": entity,
                        "wikidata_id": wikidata_id,
                        "label": label,
                        "description": description,
                        "wikidata_url": f"https://www.wikidata.org/wiki/{wikidata_id}",
                        "relevance_score": round(final_score, 3)
                    }
            
            return best_match if highest_score > 0.7 else None  # Filter low-confidence matches
    return None

def get_connected_entities(wikidata_id):
    """Retrieve all Wikidata entities directly connected to a given entity."""
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{wikidata_id}.json"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        entity_data = data.get("entities", {}).get(wikidata_id, {})
        claims = entity_data.get("claims", {})

        connected_entities = set()
        for prop, values in claims.items():
            for value in values:
                mainsnak = value.get("mainsnak", {})
                datavalue = mainsnak.get("datavalue", {})
                if datavalue.get("type") == "wikibase-entityid":
                    connected_entities.add(datavalue["value"]["id"])

        return connected_entities
    return set()

def clean_entity(entity):
    """Clean entity name by removing stop words and special characters."""
    entity = entity.lower().strip()
    entity = re.sub(r'[^\w\s]', '', entity)  # Remove punctuation
    return entity

def extract_entities(text):
    """Extract subjects and objects from the sentence."""
    doc = nlp(text)
    
    # Find subject first
    subject = None
    for token in doc:
        if token.dep_ in ("nsubj", "nsubjpass"):
            subject = token.text
            break

    # Find additional entities (proper nouns, noun chunks)
    entities = set(ent.text for ent in doc.ents)  # Named entities
    for chunk in doc.noun_chunks:
        clean_chunk = clean_entity(chunk.text)
        if clean_chunk and len(clean_chunk) > 2:
            entities.add(chunk.text)

    return subject, list(entities)

def filter_entities_by_graph(subject_wikidata_id, entities):
    """Filter entities based on direct Wikidata connections."""
    connected_entities = get_connected_entities(subject_wikidata_id)
    
    filtered_entities = []
    for entity in entities:
        result = search_wikidata(entity)
        if result and result["wikidata_id"] in connected_entities:
            filtered_entities.append(result)
    
    return filtered_entities

def extract_relationships(doc):
    """Extract subject-predicate-object triples using dependency parsing."""
    relationships = []
    for token in doc:
        if token.pos_ == "VERB":
            subj = None
            obj = None
            subj_tokens = [child for child in token.children if child.dep_ in ("nsubj", "nsubjpass")]
            if subj_tokens:
                subj = ' '.join([t.text for t in subj_tokens[0].subtree])
            
            obj_tokens = [child for child in token.children if child.dep_ in ("dobj", "attr", "prep")]
            if obj_tokens:
                obj = ' '.join([t.text for t in obj_tokens[0].subtree])
            
            if subj and obj:
                relationships.append({
                    "subject": subj,
                    "predicate": token.lemma_,
                    "object": obj
                })
    return relationships

def annotate_text(text):
    """Annotate text with entities and relationships using graph-based filtering."""
    doc = nlp(text)

    # Step 1: Extract subject first
    subject, entities = extract_entities(text)
    
    if not subject:
        return json.dumps({"error": "No subject found in sentence"}, indent=4)

    # Step 2: Get subject Wikidata entity
    subject_result = search_wikidata(subject)
    if not subject_result:
        return json.dumps({"error": f"Could not find Wikidata entity for subject: {subject}"}, indent=4)

    # Step 3: Retrieve only directly connected entities
    filtered_entities = filter_entities_by_graph(subject_result["wikidata_id"], entities)

    # Step 4: Extract relationships
    relationships = extract_relationships(doc)

    output = {
        "original_sentence": text,
        "subject": subject_result,
        "entities": filtered_entities,
        "relationships": relationships
    }
    return json.dumps(output, indent=4)

# Example Usage
if __name__ == "__main__":
    sentence = "Albert Einstein developed the theory of relativity."
    json_output = annotate_text(sentence)
    print(json_output)


{
    "original_sentence": "Albert Einstein developed the theory of relativity.",
    "subject": {
        "entity": "Einstein",
        "wikidata_id": "Q16834800",
        "label": "Einstein",
        "description": "family name",
        "wikidata_url": "https://www.wikidata.org/wiki/Q16834800",
        "relevance_score": 1.0
    },
    "entities": [],
    "relationships": [
        {
            "subject": "Albert Einstein",
            "predicate": "develop",
            "object": "the theory of relativity"
        }
    ]
}


# chebi time 


In [34]:
import json
import spacy
import rdflib
import os
import re
from difflib import SequenceMatcher

# ---------------------------
# Load spaCy Model
# ---------------------------
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_lg")
print("spaCy model loaded.")

# ---------------------------
# Load ChEBI Ontology from OWL
# ---------------------------
def load_chebi_ontology(owl_file):
    """
    Parses the ChEBI OWL file and extracts concepts with their labels, synonyms, and descriptions.
    Returns a dictionary {label/synonym: (CHEBI ID, description, relationships)}.
    """
    g = rdflib.Graph()
    
    print(f"Loading ChEBI ontology from {owl_file}...")
    g.parse(owl_file, format="xml")  # OWL files use RDF/XML format
    print("Ontology loaded.")

    chebi_concepts = {}

    for s, p, o in g:
        s, p, o = str(s), str(p), str(o)

        # Extract ChEBI ID
        if "purl.obolibrary.org/obo/CHEBI_" in s:
            chebi_id = s.split("/")[-1].replace("_", ":")

            # Extract Labels (Names)
            if p.endswith("label"):
                chebi_concepts[o.lower()] = (chebi_id, "", [])

            # Extract Synonyms
            elif p.endswith("hasExactSynonym") or p.endswith("hasRelatedSynonym"):
                chebi_concepts[o.lower()] = (chebi_id, "", [])

            # Extract Definitions
            elif p.endswith("definition") and chebi_id in chebi_concepts.values():
                chebi_concepts[o.lower()] = (chebi_id, o, [])

            # Extract Relationships (Parent/Child Links)
            elif p.endswith("is_a") or p.endswith("has_part"):
                if chebi_id in chebi_concepts:
                    chebi_concepts[o.lower()][2].append(chebi_id)

    print(f"Extracted {len(chebi_concepts)} concepts from the ontology.")
    return chebi_concepts

# ---------------------------
# String Similarity Function
# ---------------------------
def string_similarity(a, b):
    """Compute similarity between two strings using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

# ---------------------------
# Extract Entities from Text
# ---------------------------
def extract_entities(text):
    """
    Extract entities using spaCy's Named Entity Recognition (NER).
    Returns a list of detected entities.
    """
    doc = nlp(text)
    entities = set(ent.text for ent in doc.ents)
    
    # Include noun chunks as well
    for chunk in doc.noun_chunks:
        clean_chunk = re.sub(r'[^\w\s]', '', chunk.text.lower())  # Remove punctuation
        if len(clean_chunk) > 2:  # Avoid short words
            entities.add(chunk.text)

    return list(entities)

# ---------------------------
# Search ChEBI for Matching Concepts
# ---------------------------
def search_chebi(entity, chebi_dict):
    """
    Search the ChEBI dictionary for the best match.
    Returns the most relevant ChEBI concept.
    """
    best_match = None
    highest_score = 0.0

    for label, (chebi_id, description, relationships) in chebi_dict.items():
        similarity = string_similarity(entity, label)
        if similarity > highest_score:
            highest_score = similarity
            best_match = {
                "entity": entity,
                "chebi_id": chebi_id,
                "label": label,
                "description": description,
                "relationships": relationships,
                "chebi_url": f"https://www.ebi.ac.uk/chebi/searchId.do?chebiId={chebi_id}",
                "similarity_score": round(similarity, 3)
            }

    return best_match if best_match and best_match["similarity_score"] > 0.7 else None  # Threshold for quality

# ---------------------------
# Extract Relationships from Text
# ---------------------------
def extract_relationships(doc):
    """Extract subject-predicate-object triples using dependency parsing."""
    relationships = []
    for token in doc:
        if token.pos_ == "VERB":
            subj = None
            obj = None
            subj_tokens = [child for child in token.children if child.dep_ in ("nsubj", "nsubjpass")]
            if subj_tokens:
                subj = ' '.join([t.text for t in subj_tokens[0].subtree])
            
            obj_tokens = [child for child in token.children if child.dep_ in ("dobj", "attr", "prep")]
            if obj_tokens:
                obj = ' '.join([t.text for t in obj_tokens[0].subtree])
            
            if subj and obj:
                relationships.append({
                    "subject": subj,
                    "predicate": token.lemma_,
                    "object": obj
                })
    return relationships

# ---------------------------
# Annotate Text with ChEBI Concepts
# ---------------------------
def annotate_text(text, chebi_dict):
    """
    Annotate a natural language statement with ChEBI concepts.
    Returns a JSON-formatted string.
    """
    doc = nlp(text)
    entities = extract_entities(text)
    annotations = []

    # Find subject entity
    subject_entity = None
    for entity in entities:
        match = search_chebi(entity, chebi_dict)
        if match:
            annotations.append(match)
            if subject_entity is None:
                subject_entity = match

    # If a subject is found, refine other entities based on its connections
    if subject_entity:
        subject_chebi_id = subject_entity["chebi_id"]
        connected_entities = set(subject_entity["relationships"])  # Related ChEBI IDs

        refined_annotations = []
        for annotation in annotations:
            if annotation["chebi_id"] in connected_entities or annotation["chebi_id"] == subject_chebi_id:
                refined_annotations.append(annotation)
        
        annotations = refined_annotations

    # Extract Relationships
    relationships = extract_relationships(doc)

    return json.dumps({
        "original_sentence": text,
        "annotations": annotations,
        "relationships": relationships
    }, indent=4)

# ---------------------------
# Main Execution
# ---------------------------
if __name__ == "__main__":
    # Path to the ChEBI OWL file (Update this path if needed)
    chebi_owl_file = "/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl"

    if not os.path.exists(chebi_owl_file):
        print(f"Error: {chebi_owl_file} not found. Please provide the correct OWL file.")
        exit(1)

    # Load ChEBI ontology
    chebi_dict = load_chebi_ontology(chebi_owl_file)

    # Example natural language statement
    sentence = "Aspirin inhibits cyclooxygenase enzymes."
    annotated_result = annotate_text(sentence, chebi_dict)

    # Output the result
    print("Annotation Result:")
    print(annotated_result)


Loading spaCy model...
spaCy model loaded.
Loading ChEBI ontology from /home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl...
Ontology loaded.
Extracted 539370 concepts from the ontology.
Annotation Result:
{
    "original_sentence": "Aspirin inhibits cyclooxygenase enzymes.",
    "annotations": [
        {
            "entity": "Aspirin",
            "chebi_id": "CHEBI:15365",
            "label": "aspirin",
            "description": "",
            "relationships": [],
            "chebi_url": "https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:15365",
            "similarity_score": 1.0
        }
    ],
    "relationships": [
        {
            "subject": "Aspirin",
            "predicate": "inhibit",
            "object": "cyclooxygenase enzymes"
        }
    ]
}


In [35]:
import json
import spacy
import rdflib
import os
import re
from difflib import SequenceMatcher

# ---------------------------
# Load spaCy Model
# ---------------------------
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")  # Use "sm" for speed; "lg" for accuracy
print("spaCy model loaded.")

# ---------------------------
# Load ChEBI Ontology from OWL
# ---------------------------
def load_chebi_ontology(owl_file):
    """
    Parses the ChEBI OWL file and extracts concepts with their labels, synonyms, and descriptions.
    Returns a dictionary {label/synonym: (CHEBI ID, description, relationships)}.
    """
    g = rdflib.Graph()
    
    print(f"Loading ChEBI ontology from {owl_file}...")
    g.parse(owl_file, format="xml")  # OWL files use RDF/XML format
    print("Ontology loaded.")

    chebi_concepts = {}

    for s, p, o in g:
        s, p, o = str(s), str(p), str(o)

        if "purl.obolibrary.org/obo/CHEBI_" in s:
            chebi_id = s.split("/")[-1].replace("_", ":")

            if p.endswith("label"):
                chebi_concepts[o.lower()] = (chebi_id, "", [])

            elif p.endswith("hasExactSynonym") or p.endswith("hasRelatedSynonym"):
                chebi_concepts[o.lower()] = (chebi_id, "", [])

            elif p.endswith("definition"):
                if o not in chebi_concepts:
                    chebi_concepts[o.lower()] = (chebi_id, o, [])

            elif p.endswith("is_a") or p.endswith("has_part"):
                if chebi_id in chebi_concepts:
                    chebi_concepts[o.lower()][2].append(chebi_id)

    print(f"Extracted {len(chebi_concepts)} concepts.")
    return chebi_concepts

# ---------------------------
# String Similarity Function (Optimized)
# ---------------------------
def string_similarity(a, b):
    """Compute string similarity score using SequenceMatcher."""
    if a == b:
        return 1.0
    if len(a) < 3 or len(b) < 3:
        return 0.0  # Ignore very short words
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

# ---------------------------
# Extract Entities from Text (Optimized)
# ---------------------------
def extract_entities(text):
    """
    Extract entities using spaCy's Named Entity Recognition (NER).
    Returns a list of detected entities.
    """
    doc = nlp(text)
    entities = {ent.text for ent in doc.ents}
    
    for chunk in doc.noun_chunks:
        clean_chunk = re.sub(r'[^\w\s]', '', chunk.text.lower())
        if len(clean_chunk) > 2:
            entities.add(chunk.text)

    return list(entities)

# ---------------------------
# Search ChEBI for Matching Concepts (Optimized)
# ---------------------------
def search_chebi(entity, chebi_dict):
    """
    Search the ChEBI dictionary for the best match.
    Uses direct lookup first, then applies fuzzy matching.
    """
    entity_clean = entity.lower()
    
    # Direct lookup first
    if entity_clean in chebi_dict:
        chebi_id, description, relationships = chebi_dict[entity_clean]
        return {
            "entity": entity,
            "chebi_id": chebi_id,
            "label": entity,
            "description": description,
            "relationships": relationships,
            "chebi_url": f"https://www.ebi.ac.uk/chebi/searchId.do?chebiId={chebi_id}",
            "similarity_score": 1.0
        }

    # Fuzzy matching fallback
    best_match = None
    highest_score = 0.7  # Only accept strong matches

    for label, (chebi_id, description, relationships) in chebi_dict.items():
        similarity = string_similarity(entity_clean, label)
        if similarity > highest_score:
            highest_score = similarity
            best_match = {
                "entity": entity,
                "chebi_id": chebi_id,
                "label": label,
                "description": description,
                "relationships": relationships,
                "chebi_url": f"https://www.ebi.ac.uk/chebi/searchId.do?chebiId={chebi_id}",
                "similarity_score": round(similarity, 3)
            }

    return best_match

# ---------------------------
# Extract Relationships from Text (Optimized)
# ---------------------------
def extract_relationships(doc):
    """Extract subject-predicate-object triples using dependency parsing."""
    relationships = []
    for token in doc:
        if token.pos_ == "VERB":
            subj = None
            obj = None
            subj_tokens = [child for child in token.children if child.dep_ in ("nsubj", "nsubjpass")]
            if subj_tokens:
                subj = ' '.join([t.text for t in subj_tokens[0].subtree])
            
            obj_tokens = [child for child in token.children if child.dep_ in ("dobj", "attr", "prep")]
            if obj_tokens:
                obj = ' '.join([t.text for t in obj_tokens[0].subtree])
            
            if subj and obj:
                relationships.append({
                    "subject": subj,
                    "predicate": token.lemma_,
                    "object": obj
                })
    return relationships

# ---------------------------
# Annotate Text with ChEBI Concepts (Optimized)
# ---------------------------
def annotate_text(text, chebi_dict):
    """
    Annotate a natural language statement with ChEBI concepts.
    Returns a JSON-formatted string.
    """
    doc = nlp(text)
    entities = extract_entities(text)
    annotations = []

    # Step 1: Find subject entity
    subject_entity = None
    for entity in entities:
        match = search_chebi(entity, chebi_dict)
        if match:
            annotations.append(match)
            if subject_entity is None:
                subject_entity = match

    # Step 2: Filter entities based on subject relationships
    if subject_entity:
        subject_chebi_id = subject_entity["chebi_id"]
        connected_entities = set(subject_entity["relationships"])

        annotations = [annotation for annotation in annotations if annotation["chebi_id"] in connected_entities or annotation["chebi_id"] == subject_chebi_id]

    # Step 3: Extract Relationships
    relationships = extract_relationships(doc)

    return json.dumps({
        "original_sentence": text,
        "annotations": annotations,
        "relationships": relationships
    }, indent=4)

# ---------------------------
# Main Execution (Optimized)
# ---------------------------
if __name__ == "__main__":
    chebi_owl_file = "/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl"

    if not os.path.exists(chebi_owl_file):
        print(f"Error: {chebi_owl_file} not found. Please provide the correct OWL file.")
        exit(1)

    print("Preloading ChEBI ontology...")
    chebi_dict = load_chebi_ontology(chebi_owl_file)

    print("Running annotation...")
    sentence = "Aspirin inhibits cyclooxygenase enzymes."
    annotated_result = annotate_text(sentence, chebi_dict)

    print("Annotation Result:")
    print(annotated_result)


Loading spaCy model...
spaCy model loaded.
Preloading ChEBI ontology...
Loading ChEBI ontology from /home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl...
Ontology loaded.
Extracted 539370 concepts.
Running annotation...
Annotation Result:
{
    "original_sentence": "Aspirin inhibits cyclooxygenase enzymes.",
    "annotations": [
        {
            "entity": "cyclooxygenase enzymes",
            "chebi_id": "CHEBI:35544",
            "label": "cyclooxygenase inhibitors",
            "description": "",
            "relationships": [],
            "chebi_url": "https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:35544",
            "similarity_score": 0.723
        }
    ],
    "relationships": []
}
